# Build a churn classifier and evaluate it with the right metric

One row is one telco customer; `Churn` is `Yes` when the customer left in the last month. The retention manager: "My team calls the customers you flag and offers a discount. A call costs us little. A customer who leaves costs us a year of bills."

## The starter

The cell repeats D01 5.6: `TotalCharges` converted to numbers, the `ColumnTransformer` `pre`, and the 80 / 20 split with `stratify=y, random_state=42`, so your 1,409 test customers are the same as in the 4.2 demonstration.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print(len(X_train), "training customers,", len(X_test), "test customers,", y_test.sum(), "test churners")

## Step 1 · Fit and score against a dummy

Fit `make_pipeline(pre, LogisticRegression(max_iter=1000))`. Print the test accuracy and the accuracy of `DummyClassifier(strategy="most_frequent")`.

## Step 2 · The confusion matrix

Print `confusion_matrix(y_test, model.predict(X_test))`. Write down what each of the four cells counts, in words: "customers who stayed and whom we did not flag", and so on.

## Step 3 · Precision, recall and F1

Print `classification_report(y_test, pred, digits=3)`. Find precision, recall and F1 for the churn class (`1`), and check precision and recall by hand from your four cells.

## Step 4 · Ranking metrics

Compute `roc_auc_score` and `average_precision_score` from `model.predict_proba(X_test)[:, 1]`. Compare the average precision with the share of churners in the test rows.

## Step 5 · A second model with `class_weight="balanced"`

Fit `LogisticRegression(max_iter=1000, class_weight="balanced")` in the same pipeline. Repeat steps 2 to 4 for it.

## Step 6 · "Which model should my team use?"

Translate: which of her two costs is larger, a wasted call (a false positive) or a missed churner (a false negative)? Which metric counts the misses? Answer in two sentences with the numbers from both models.

## Check yourself